# CallWhisper-8k: OpenAI Whisper GPU Benchmark

Purpose: reproduce the local `small` result on Colab, then run stronger OpenAI Whisper models on the same fixed GramVaani slices.

Expected outputs go into `results/colab_*`. Do not upload raw audio to GitHub.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/anshulLuhsna/CallWhisper-8k.git"
REPO_DIR = Path("/content/CallWhisper-8k")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/call-whisper")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = "src"
print("Repo:", Path.cwd())
print("Drive project dir:", DRIVE_PROJECT_DIR)


In [ ]:
!nvidia-smi
!python -m pip install -U pip
!python -m pip install -e ".[dev]"
!apt-get update -qq && apt-get install -y -qq ffmpeg
os.environ["PYTHONPATH"] = "src"

## Mount Or Locate Audio

The committed manifests use repo-relative paths like `datasets/GV_Dev_5h/Audio/...`. This cell checks that those files exist. If your audio is in Google Drive, mount Drive and copy/symlink it into the expected repo path.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

if not DRIVE_PROJECT_DIR.exists():
    raise FileNotFoundError(f"Expected Drive folder not found: {DRIVE_PROJECT_DIR}")

def link_or_replace(source: Path, target: Path) -> None:
    if not source.exists():
        raise FileNotFoundError(f"Missing Drive source: {source}")
    if target.exists() or target.is_symlink():
        return
    target.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(source, target)

link_or_replace(DRIVE_PROJECT_DIR / "GV_Dev_5h", REPO_DIR / "datasets/GV_Dev_5h")
link_or_replace(DRIVE_PROJECT_DIR / "Metadata", REPO_DIR / "datasets/Metadata")

drive_manifest_dir = DRIVE_PROJECT_DIR / "manifests"
repo_manifest_dir = REPO_DIR / "datasets/manifests"
if drive_manifest_dir.exists():
    repo_manifest_dir.mkdir(parents=True, exist_ok=True)
    for manifest in drive_manifest_dir.glob("*.csv"):
        shutil.copy2(manifest, repo_manifest_dir / manifest.name)

expected_audio_dir = REPO_DIR / "datasets/GV_Dev_5h/Audio"
print("Drive project dir:", DRIVE_PROJECT_DIR)
print("Audio dir:", expected_audio_dir)
print("Audio exists:", expected_audio_dir.exists())
print("MP3 files:", len(list(expected_audio_dir.glob("*.mp3"))) if expected_audio_dir.exists() else 0)
print("Manifest files:", len(list(repo_manifest_dir.glob("*.csv"))))


In [ ]:
import csv

manifests = [
    "datasets/manifests/gramvaani_dev_50.csv",
    "datasets/manifests/gramvaani_dev_50_8khz.csv",
    "datasets/manifests/gramvaani_dev_50_highrate.csv",
]

for manifest in manifests:
    missing = []
    with open(manifest, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            if not Path(row["audio_path"]).exists():
                missing.append(row["audio_path"])
    print(manifest, "missing", len(missing))
    if missing[:3]:
        print("examples:", missing[:3])

## Run OpenAI Whisper

Start with `small` to reproduce the local reference. Then run `medium`. Add `large-v3` only if the Colab GPU/runtime can handle it.

In [ ]:
import subprocess

runs = [
    ("small", "datasets/manifests/gramvaani_dev_50.csv", "results/colab_whisper_small_gramvaani_dev_50_seed0.json"),
    ("medium", "datasets/manifests/gramvaani_dev_50.csv", "results/colab_whisper_medium_gramvaani_dev_50_seed0.json"),
    ("medium", "datasets/manifests/gramvaani_dev_50_8khz.csv", "results/colab_whisper_medium_gramvaani_dev_50_8khz_seed0.json"),
    ("medium", "datasets/manifests/gramvaani_dev_50_highrate.csv", "results/colab_whisper_medium_gramvaani_dev_50_highrate_seed0.json"),
]

for model, manifest, output_json in runs:
    cmd = [
        sys.executable, "-m", "callwhisper.eval",
        "--manifest", manifest,
        "--model", model,
        "--language-mode", "manifest",
        "--seed", "0",
        "--output-json", output_json,
    ]
    print("RUN", " ".join(cmd))
    subprocess.run(cmd, check=True, env={**os.environ, "PYTHONPATH": "src"})

In [ ]:
import json, pandas as pd

rows = []
for path in sorted(Path("results").glob("colab_whisper_*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))
    sample0 = payload["samples"][0]
    rows.append({
        "file": str(path),
        "model": sample0["model"],
        "slice": sample0["slice"],
        "condition": sample0["condition"],
        "files": payload["summary"]["num_files"],
        "wer": round(payload["summary"]["wer"], 4),
        "cer": round(payload["summary"]["cer"], 4),
    })

df = pd.DataFrame(rows).sort_values(["model", "slice"])
df

In [ ]:
Path("results/colab_openai_whisper_summary.md").write_text(df.to_markdown(index=False) + "\n", encoding="utf-8")
print(Path("results/colab_openai_whisper_summary.md").read_text(encoding="utf-8"))